
# Sequence Forecasting Framework: LSTM, Bi-LSTM, and Attention Bi-LSTM

This notebook extends the refactored experiment framework to support three model families:

- **LSTM**
- **Bi-LSTM**
- **Bi-LSTM + Attention**

It keeps the same reusable structure:

- configurable data loading
- lag feature generation
- automatic sequence rebuilding when `lookback` changes
- reusable training / evaluation
- single run or parameter sweep
- result saving and best-model review
- optional attention-weight visualisation

It is designed to run either:

- **locally**
- in **Google Colab**
- from a **cloned GitHub repo** with data stored on Google Drive


In [1]:
#!git clone https://github.com/UNSW-ZZSC9020/project.git /content/capstone_project_GroupA

In [2]:

import os
import copy
import json
import random
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [3]:

# 1. PATHS + SETTINGS
#


SEED = 42
RUN_SWEEP = True
SHOW_LIVE_PLOTS = False
SAVE_RESULTS = True
SAVE_BEST_MODEL = True

# There was a few discrpencies when I first downloaded the repository. Probsbly no longer eneded.


LOCAL_REPO_CANDIDATES = [
    "capstone_project_GroupA",
    "/content/capstone_project_GroupA",
]

LOCAL_DATA_CANDIDATES = [
    "data/NSW",
    "capstone_project_GroupA/data/NSW",
    "/content/capstone_project_GroupA/data/NSW",
    "/content/drive/MyDrive/capstone_project_GroupA/data/NSW",
]

def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

REPO_PATH = first_existing(LOCAL_REPO_CANDIDATES)
DATA_DIR = first_existing(LOCAL_DATA_CANDIDATES)

if DATA_DIR is None:
    print("Could not auto-detect DATA_DIR.")
    print("Set DATA_DIR manually before running the training cells.")
else:
    print("Using DATA_DIR:", DATA_DIR)

if REPO_PATH is None:
    REPO_PATH = os.getcwd()

OUTPUT_DIR = os.path.join(REPO_PATH, "output", "sequence_model_experiments")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_SCALED_PATH = os.path.join(DATA_DIR, "train_scaled.csv") if DATA_DIR else None
VAL_SCALED_PATH   = os.path.join(DATA_DIR, "val_scaled.csv") if DATA_DIR else None
TEST_SCALED_PATH  = os.path.join(DATA_DIR, "test_scaled.csv") if DATA_DIR else None

TRAIN_RAW_PATH    = os.path.join(DATA_DIR, "train.csv") if DATA_DIR else None
VAL_RAW_PATH      = os.path.join(DATA_DIR, "validation.csv") if DATA_DIR else None
TEST_RAW_PATH     = os.path.join(DATA_DIR, "test.csv") if DATA_DIR else None

BASE_CONFIG = {
    "model_type": "multihead_attention_bilstm",
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.4,
    "learning_rate": 5e-5,
    "batch_size": 64,
    "epochs": 100,
    "patience": 8,
    "use_mlp_head": True,
    "mlp_hidden_size": 64,
    "target_col": "TOTALDEMAND",
    "temp_col": "TEMPERATURE",
    "demand_lags": [],
    "temp_lags": [0,2,50],
    "lookback": 168*5,
    "horizon": 168,
    "weight_decay": 1e-4,
    "num_attention_heads": 4,
}

print("Output directory:", OUTPUT_DIR)
print("Base config:")
for k, v in BASE_CONFIG.items():
    print(f"  {k}: {v}")


Using DATA_DIR: capstone_project_GroupA/data/NSW
Output directory: capstone_project_GroupA/output/sequence_model_experiments
Base config:
  model_type: multihead_attention_bilstm
  hidden_size: 64
  num_layers: 2
  dropout: 0.4
  learning_rate: 5e-05
  batch_size: 64
  epochs: 100
  patience: 8
  use_mlp_head: True
  mlp_hidden_size: 64
  target_col: TOTALDEMAND
  temp_col: TEMPERATURE
  demand_lags: []
  temp_lags: [0, 2, 50]
  lookback: 840
  horizon: 168
  weight_decay: 0.0001
  num_attention_heads: 4


In [4]:
# Making sure the seed is consistent
def set_seed(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [5]:

# Date prep helpers

def add_lag_features(
    df,
    target_col="TOTALDEMAND",
    temp_col="TEMPERATURE",
    demand_lags=None,
    temp_lags=None,
):
    df = df.copy()
    demand_lags = demand_lags or []
    temp_lags = temp_lags or []

    for lag in demand_lags:
        df[f"demand_lag_{lag}"] = df[target_col].shift(lag)

    if temp_col in df.columns:
        for lag in temp_lags:
            df[f"temp_lag_{lag}"] = df[temp_col].shift(lag)

    df = df.dropna().reset_index(drop=True)
    return df


def validate_required_files():
    required = [
        TRAIN_SCALED_PATH, VAL_SCALED_PATH, TEST_SCALED_PATH,
        TRAIN_RAW_PATH, VAL_RAW_PATH, TEST_RAW_PATH
    ]
    missing = [p for p in required if p is None or not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Missing one or more required CSV files:\n" + "\n".join(str(m) for m in missing)
        )


def load_data(config):
    validate_required_files()

    train_scaled_df = pd.read_csv(TRAIN_SCALED_PATH)
    val_scaled_df   = pd.read_csv(VAL_SCALED_PATH)
    test_scaled_df  = pd.read_csv(TEST_SCALED_PATH)

    train_raw_df = pd.read_csv(TRAIN_RAW_PATH)
    val_raw_df   = pd.read_csv(VAL_RAW_PATH)
    test_raw_df  = pd.read_csv(TEST_RAW_PATH)

    train_scaled_df = add_lag_features(
        train_scaled_df,
        target_col=config["target_col"],
        temp_col=config["temp_col"],
        demand_lags=config["demand_lags"],
        temp_lags=config["temp_lags"],
    )
    val_scaled_df = add_lag_features(
        val_scaled_df,
        target_col=config["target_col"],
        temp_col=config["temp_col"],
        demand_lags=config["demand_lags"],
        temp_lags=config["temp_lags"],
    )
    test_scaled_df = add_lag_features(
        test_scaled_df,
        target_col=config["target_col"],
        temp_col=config["temp_col"],
        demand_lags=config["demand_lags"],
        temp_lags=config["temp_lags"],
    )

    train_raw_df = add_lag_features(
        train_raw_df,
        target_col=config["target_col"],
        temp_col=config["temp_col"],
        demand_lags=config["demand_lags"],
        temp_lags=config["temp_lags"],
    )
    val_raw_df = add_lag_features(
        val_raw_df,
        target_col=config["target_col"],
        temp_col=config["temp_col"],
        demand_lags=config["demand_lags"],
        temp_lags=config["temp_lags"],
    )
    test_raw_df = add_lag_features(
        test_raw_df,
        target_col=config["target_col"],
        temp_col=config["temp_col"],
        demand_lags=config["demand_lags"],
        temp_lags=config["temp_lags"],
    )

    lag_feature_cols = (
        [f"demand_lag_{lag}" for lag in config["demand_lags"]] +
        [f"temp_lag_{lag}" for lag in config["temp_lags"]]
    )

    multivar_cols = [config["target_col"], config["temp_col"]] + lag_feature_cols

    train_target_scaled = train_scaled_df[[config["target_col"]]].values.astype(np.float32)
    val_target_scaled   = val_scaled_df[[config["target_col"]]].values.astype(np.float32)
    test_target_scaled  = test_scaled_df[[config["target_col"]]].values.astype(np.float32)

    train_x = train_scaled_df[multivar_cols].values.astype(np.float32)
    val_x   = val_scaled_df[multivar_cols].values.astype(np.float32)
    test_x  = test_scaled_df[multivar_cols].values.astype(np.float32)

    train_y = train_target_scaled.ravel()
    val_y   = val_target_scaled.ravel()
    test_y  = test_target_scaled.ravel()

    target_scaler = StandardScaler()
    target_scaler.fit(train_raw_df[[config["target_col"]]].values.astype(np.float32))

    return {
        "train_x": train_x,
        "val_x": val_x,
        "test_x": test_x,
        "train_y": train_y,
        "val_y": val_y,
        "test_y": test_y,
        "target_scaler": target_scaler,
        "lag_feature_cols": lag_feature_cols,
        "multivar_cols": multivar_cols,
        "train_scaled_df": train_scaled_df,
        "val_scaled_df": val_scaled_df,
        "test_scaled_df": test_scaled_df,
        "target_scaler": target_scaler,
    }


def make_sequences(x, y, lookback, horizon):
    X_seq, y_seq = [], []
    for i in range(lookback, len(x) - horizon + 1):
        X_seq.append(x[i - lookback:i])
        y_seq.append(y[i + horizon - 1])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)


class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def build_dataloaders(data_dict, config):
    X_train, y_train = make_sequences(
        data_dict["train_x"], data_dict["train_y"], config["lookback"], config["horizon"]
    )
    X_val, y_val = make_sequences(
        data_dict["val_x"], data_dict["val_y"], config["lookback"], config["horizon"]
    )
    X_test, y_test = make_sequences(
        data_dict["test_x"], data_dict["test_y"], config["lookback"], config["horizon"]
    )

    train_loader = DataLoader(
        SequenceDataset(X_train, y_train),
        batch_size=config["batch_size"],
        shuffle=True,
    )
    val_loader = DataLoader(
        SequenceDataset(X_val, y_val),
        batch_size=config["batch_size"],
        shuffle=False,
    )
    test_loader = DataLoader(
        SequenceDataset(X_test, y_test),
        batch_size=config["batch_size"],
        shuffle=False,
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
    }


In [6]:

# Models
class SequenceForecaster(nn.Module):
    """
    Plain LSTM / Bi-LSTM forecaster using the final timestep output.
    Optionally uses an MLP head with ReLU.
    """
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        dropout=0.0,
        bidirectional=False,
        use_mlp_head=False,
        mlp_hidden_size=32,
    ):
        super().__init__()

        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.use_mlp_head = use_mlp_head

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
            bidirectional=bidirectional,
        )

        self.dropout = nn.Dropout(dropout)

        output_dim = hidden_size * self.num_directions

        if use_mlp_head:
            self.fc = nn.Sequential(
                nn.Linear(output_dim, mlp_hidden_size),
                nn.ReLU(),
                nn.Linear(mlp_hidden_size, 1),
            )
        else:
            self.fc = nn.Linear(output_dim, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        last_out = self.dropout(last_out)
        out = self.fc(last_out)
        return out.view(-1)


class AttentionBiLSTMForecaster(nn.Module):
    """
    Bi-LSTM with attention over timesteps.
    Optionally uses an MLP head with ReLU.
    """
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        dropout=0.0,
        bidirectional=True,
        use_mlp_head=False,
        mlp_hidden_size=32,
    ):
        super().__init__()

        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.use_mlp_head = use_mlp_head

        lstm_dropout = dropout if num_layers > 1 else 0.0
        output_dim = hidden_size * self.num_directions

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
            bidirectional=bidirectional,
        )

        self.attention = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(dropout)

        if use_mlp_head:
            self.fc = nn.Sequential(
                nn.Linear(output_dim, mlp_hidden_size),
                nn.ReLU(),
                nn.Linear(mlp_hidden_size, 1),
            )
        else:
            self.fc = nn.Linear(output_dim, 1)

    def forward(self, x, return_attention=False):
        lstm_out, _ = self.lstm(x)

        # Optional scaling (stability improvement)
        attn_scores = self.attention(lstm_out) / (lstm_out.size(-1) ** 0.5)
        attn_weights = torch.softmax(attn_scores, dim=1)

        context = torch.sum(attn_weights * lstm_out, dim=1)
        context = self.dropout(context)

        out = self.fc(context).view(-1)

        if return_attention:
            return out, attn_weights.squeeze(-1)

        return out

import torch
import torch.nn as nn


class MultiHeadAttentionBiLSTMForecaster(nn.Module):
    """
    Bi-LSTM with multi-head temporal attention over timesteps.
    Each head learns a separate attention distribution over the sequence.
    """
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        dropout=0.0,
        bidirectional=True,
        use_mlp_head=False,
        mlp_hidden_size=32,
        num_attention_heads=4,
    ):
        super().__init__()

        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.num_attention_heads = num_attention_heads

        lstm_dropout = dropout if num_layers > 1 else 0.0
        output_dim = hidden_size * self.num_directions

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
            bidirectional=bidirectional,
        )

        # one score per head per timestep
        self.attention = nn.Linear(output_dim, num_attention_heads)
        self.dropout = nn.Dropout(dropout)

        combined_dim = output_dim * num_attention_heads

        if use_mlp_head:
            self.fc = nn.Sequential(
                nn.Linear(combined_dim, mlp_hidden_size),
                nn.ReLU(),
                nn.Linear(mlp_hidden_size, 1),
            )
        else:
            self.fc = nn.Linear(combined_dim, 1)

    def forward(self, x, return_attention=False):
        # lstm_out: (batch, seq_len, output_dim)
        lstm_out, _ = self.lstm(x)

        # attn_scores: (batch, seq_len, num_heads)
        attn_scores = self.attention(lstm_out)

        # softmax over time dimension
        attn_weights = torch.softmax(attn_scores, dim=1)

        # reshape for broadcasting
        # lstm_out_expanded: (batch, seq_len, 1, output_dim)
        lstm_out_expanded = lstm_out.unsqueeze(2)

        # attn_weights_expanded: (batch, seq_len, num_heads, 1)
        attn_weights_expanded = attn_weights.unsqueeze(-1)

        # weighted sum over time
        # context: (batch, num_heads, output_dim)
        context = torch.sum(attn_weights_expanded * lstm_out_expanded, dim=1)

        # flatten heads
        # context: (batch, num_heads * output_dim)
        context = context.reshape(context.size(0), -1)
        context = self.dropout(context)

        out = self.fc(context).view(-1)

        if return_attention:
            # return attention as (batch, num_heads, seq_len) for easier plotting
            return out, attn_weights.permute(0, 2, 1)

        return out


def build_model(config, input_size):
    model_type = config["model_type"]

    if model_type == "lstm":
        model = SequenceForecaster(
            input_size=input_size,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
            dropout=config["dropout"],
            bidirectional=False,
            use_mlp_head=config.get("use_mlp_head", False),
            mlp_hidden_size=config.get("mlp_hidden_size", 32),
        )

    elif model_type == "bilstm":
        model = SequenceForecaster(
            input_size=input_size,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
            dropout=config["dropout"],
            bidirectional=True,
            use_mlp_head=config.get("use_mlp_head", False),
            mlp_hidden_size=config.get("mlp_hidden_size", 32),
        )

    elif model_type == "attention_bilstm":
        model = AttentionBiLSTMForecaster(
            input_size=input_size,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
            dropout=config["dropout"],
            bidirectional=True,
            use_mlp_head=config.get("use_mlp_head", False),
            mlp_hidden_size=config.get("mlp_hidden_size", 32),
        )

    elif model_type == "multihead_attention_bilstm":
        model = MultiHeadAttentionBiLSTMForecaster(
            input_size=input_size,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
            dropout=config["dropout"],
            bidirectional=True,
            use_mlp_head=config.get("use_mlp_head", False),
            mlp_hidden_size=config.get("mlp_hidden_size", 32),
            num_attention_heads=config.get("num_attention_heads", 4),
        )

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    return model.to(DEVICE)


In [7]:

# Training helpers

def live_plot_losses(train_losses, val_losses, title="Training History"):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=50,
    patience=10,
    scheduler=None,
    show_live_plots=False,
    title="Training History",
):
    model.to(device)

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_epoch = -1
    best_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * X_batch.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)

                running_val_loss += loss.item() * X_batch.size(0)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(epoch_val_loss)
            else:
                scheduler.step()

        # Stops the training when the val loss meets a certain condition
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {epoch_train_loss:.6f} | "
            f"Val Loss: {epoch_val_loss:.6f}"
        )

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    if show_live_plots:
        plot_training_history(train_losses, val_losses, title=title)

    return {
        "model": model,
        "best_state": best_state,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "train_losses": train_losses,
        "val_losses": val_losses,
    }

In [8]:

# Evaluation


def evaluate_model(model, test_loader, target_scaler, device, tolerance_pct=10.0):
    model.eval()
    y_pred_scaled = []
    y_true_scaled = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy()

            y_pred_scaled.extend(preds)
            y_true_scaled.extend(y_batch.numpy())

    y_pred_scaled = np.array(y_pred_scaled).reshape(-1, 1)
    y_true_scaled = np.array(y_true_scaled).reshape(-1, 1)

    y_pred_real = target_scaler.inverse_transform(y_pred_scaled).ravel()
    y_true_real = target_scaler.inverse_transform(y_true_scaled).ravel()

    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mae = mean_absolute_error(y_true_real, y_pred_real)
    r2 = r2_score(y_true_real, y_pred_real)

    mask = np.abs(y_true_real) > 1e-8
    mape = (
        np.mean(np.abs((y_true_real[mask] - y_pred_real[mask]) / y_true_real[mask])) * 100
        if np.any(mask) else np.nan
    )

    tolerance = tolerance_pct / 100.0
    within_tol_acc = np.mean(
        np.abs(y_pred_real - y_true_real) <= tolerance * np.abs(y_true_real)
    ) * 100

    return {
        "y_pred_scaled": y_pred_scaled,
        "y_true_scaled": y_true_scaled,
        "y_pred_real": y_pred_real,
        "y_true_real": y_true_real,
        "rmse": rmse,
        "mae": mae,
        "mape": mape,
        "r2": r2,
        "within_tol_acc": within_tol_acc,
    }


def plot_training_history(train_losses, val_losses, title="Training History"):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss (scaled)")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_predictions(y_true_real, y_pred_real, n_points=500, title="Forecast vs Actual"):
    plt.figure(figsize=(12, 5))
    plt.plot(y_true_real[:n_points], label="Actual")
    plt.plot(y_pred_real[:n_points], label="Predicted")
    plt.xlabel("Test Sample")
    plt.ylabel("Demand")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [9]:

def merge_config(base_config, overrides=None):
    cfg = copy.deepcopy(base_config)
    if overrides:
        cfg.update(overrides)
    return cfg


def make_config_name(cfg):
    demand_lags_str = "-".join(map(str, cfg.get("demand_lags", []))) if cfg.get("demand_lags") else "none"
    temp_lags_str = "-".join(map(str, cfg.get("temp_lags", []))) if cfg.get("temp_lags") else "none"

    name = (
        f"{cfg['model_type']}"
        f"_lb{cfg['lookback']}"
        f"_hs{cfg['hidden_size']}"
        f"_nl{cfg['num_layers']}"
        f"_do{cfg['dropout']}"
        f"_lr{cfg['learning_rate']}"
        f"_mlp{int(cfg.get('use_mlp_head', False))}"
        f"_dlags{demand_lags_str}"
        f"_tlags{temp_lags_str}"
    )

    if "num_attention_heads" in cfg:
        name += f"_heads{cfg['num_attention_heads']}"

    return name


def build_config_list(base_config, param_grid=None):
    """
    Builds a list of config dictionaries from a base config and optional param grid.
    If param_grid is None or empty, returns [base_config].
    """
    if not param_grid:
        return [copy.deepcopy(base_config)]

    keys = list(param_grid.keys())
    values = list(param_grid.values())

    configs = []
    for combo in product(*values):
        cfg = copy.deepcopy(base_config)
        for k, v in zip(keys, combo):
            cfg[k] = v
        configs.append(cfg)

    return configs


def run_experiment(config, experiment_name=None, save_best_model=True, show_live_plots=False):
    """
    Atomic run function.
    Assumes the following already exist earlier in the notebook:
      - SEED
      - DEVICE
      - OUTPUT_DIR
      - set_seed
      - load_data
      - build_dataloaders
      - build_model
      - train_model
      - evaluate_model
      - plot_training_history
      - plot_predictions
      - nn (torch.nn)
    """
    experiment_name = experiment_name or make_config_name(config)

    print("\n" + "=" * 90)
    print(f"Running experiment: {experiment_name}")
    print("=" * 90)
    for k, v in config.items():
        print(f"{k}: {v}")

    run_seed = config.get("seed", SEED)
    set_seed(run_seed)
    print(f"Using run seed: {run_seed}")

    data_dict = load_data(config)
    seq_dict = build_dataloaders(data_dict, config)

    # DEBUG: Inspect input features

    features = data_dict.get("features", None)

    print("\n" + "="*60)
    print("DEBUG: FEATURE INSPECTION")
    print("="*60)

    # 1. Feature names
    if features is not None:
        print("\nFeature names:")
        for i, f in enumerate(features):
            print(f"{i}: {f}")
    else:
        print("\nNo feature list found in data_dict")

    # 2. Shape check
    X_sample = seq_dict["X_train"][0]
    print("\nSequence shape (lookback, num_features):", X_sample.shape)

    # 3. Convert to DataFrame for readability
    import pandas as pd

    if features is not None:
        df_seq = pd.DataFrame(X_sample, columns=features)

        print("\nFirst 5 timesteps:")
        display(df_seq.head())

        print("\nLast 5 timesteps:")
        display(df_seq.tail())
    else:
        print("\nRaw sequence values (first 5 timesteps):")
        print(X_sample[:5])

    # 4. Target check
    y_sample = seq_dict["y_train"][0]
    print("\nTarget (scaled):", y_sample)

    # Optional: inverse transform
    if "target_scaler" in data_dict:
        y_real = data_dict["target_scaler"].inverse_transform(
            np.array(y_sample).reshape(-1, 1)
        )
        print("Target (real scale):", y_real.flatten()[0])

    print("="*60 + "\n")

    print("\nSequence shapes:")
    print("X_train:", seq_dict["X_train"].shape, "| y_train:", seq_dict["y_train"].shape)
    print("X_val:  ", seq_dict["X_val"].shape,   "| y_val:  ", seq_dict["y_val"].shape)
    print("X_test: ", seq_dict["X_test"].shape,  "| y_test: ", seq_dict["y_test"].shape)

    model = build_model(config, input_size=seq_dict["X_train"].shape[2])

    # Quick debug checks
    for name, param in model.named_parameters():
        print(f"[DEBUG] First parameter tensor: {name}")
        print("[DEBUG] First 5 values:", param.detach().view(-1)[:5].cpu().numpy())
        break

    xb0, yb0 = next(iter(seq_dict["train_loader"]))
    print("[DEBUG] First 5 y values from first train batch:", yb0[:5].cpu().numpy())

    criterion = nn.MSELoss()

    # Optimizer
    optimizer_name = config.get("optimizer", "adam").lower()
    lr = config["learning_rate"]
    weight_decay = config.get("weight_decay", 0.0)

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    scheduler = None
    scheduler_name = config.get("scheduler", None)
    if scheduler_name:
        scheduler_name = scheduler_name.lower()
        if scheduler_name == "reduce_on_plateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=config.get("scheduler_factor", 0.5),
                patience=config.get("scheduler_patience", 5),
            )
        elif scheduler_name == "step":
            scheduler = torch.optim.lr_scheduler.StepLR(
                optimizer,
                step_size=config.get("scheduler_step_size", 10),
                gamma=config.get("scheduler_gamma", 0.5),
            )
        else:
            raise ValueError(f"Unsupported scheduler: {scheduler_name}")

    train_output = train_model(
        model=model,
        train_loader=seq_dict["train_loader"],
        val_loader=seq_dict["val_loader"],
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epochs=config.get("epochs", 50),
        patience=config.get("patience", 10),
        scheduler=scheduler,
        show_live_plots=show_live_plots,
    )

    train_losses = train_output["train_losses"]
    val_losses = train_output["val_losses"]
    best_state = train_output["best_state"]
    best_epoch = train_output["best_epoch"]
    best_val_loss = train_output["best_val_loss"]

    model.load_state_dict(best_state)

    eval_dict = evaluate_model(
        model=model,
        test_loader=seq_dict["test_loader"],
        target_scaler=data_dict["target_scaler"],
        device=DEVICE,
        tolerance_pct=10.0,
    )

    result = {
        "experiment_name": experiment_name,
        "seed": run_seed,
        "best_epoch": best_epoch,
        "best_val_loss": float(best_val_loss) if best_val_loss is not None else np.nan,
        "test_rmse": eval_dict.get("rmse"),
        "test_mae": eval_dict.get("mae"),
        "test_mape": eval_dict.get("mape"),
        "test_r2": eval_dict.get("r2"),
        "test_acc_within_10pct": eval_dict.get("within_tol_acc"),
    }

    # Add selected config parameters into result row for easier sorting/filtering
    for k in [
        "model_type",
        "hidden_size",
        "num_layers",
        "dropout",
        "learning_rate",
        "use_mlp_head",
        "mlp_hidden_size",
        "lookback",
        "horizon",
        "demand_lags",
        "temp_lags",
        "num_attention_heads",
    ]:
        if k in config:
            result[k] = config[k]

    # Optional checkpoint save
    if save_best_model:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        model_path = os.path.join(OUTPUT_DIR, f"{experiment_name}_best.pt")
        torch.save(best_state, model_path)
        print(f"Saved best model state to: {model_path}")

    artifact = {
        "config": copy.deepcopy(config),
        "result": result,
        "model_class": model.__class__.__name__,
        "input_size": seq_dict["X_train"].shape[2],
        "train_losses": train_losses,
        "val_losses": val_losses,
        "best_state": copy.deepcopy(best_state),
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "eval_dict": eval_dict,
    }

    return artifact


def run_experiment_suite(
    base_config,
    param_grid=None,
    n_repeats=1,
    base_seed=42,
    save_dir=None,
    save_best_model=False,
    show_live_plots=False,
):
    """
    Unified runner for:
      - single experiment (param_grid=None, n_repeats=1)
      - parameter sweep (param_grid=..., n_repeats=1)
      - repeated experiments (param_grid=..., n_repeats>1)
    """
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    config_list = build_config_list(base_config, param_grid)
    total_runs = len(config_list) * n_repeats

    all_rows = []
    all_artifacts = []

    run_counter = 0
    print(f"Total configurations: {len(config_list)}")
    print(f"Repeats per configuration: {n_repeats}")
    print(f"Total runs: {total_runs}")

    for cfg_idx, cfg in enumerate(config_list, start=1):
        config_name = make_config_name(cfg)
        print(f"\n{'='*100}")
        print(f"[CONFIG {cfg_idx}/{len(config_list)}] {config_name}")
        print(f"{'='*100}")

        for repeat_idx in range(n_repeats):
            run_counter += 1
            run_seed = base_seed + repeat_idx

            cfg_run = copy.deepcopy(cfg)
            cfg_run["seed"] = run_seed

            experiment_name = config_name if n_repeats == 1 else f"{config_name}__seed{run_seed}"

            print(f"\nRun {run_counter}/{total_runs} | repeat {repeat_idx + 1}/{n_repeats} | seed={run_seed}")

            artifact = run_experiment(
                config=cfg_run,
                experiment_name=experiment_name,
                save_best_model=save_best_model,
                show_live_plots=show_live_plots,
            )

            result = artifact.get("result", {})
            eval_dict = artifact.get("eval_dict", {})

            row = {
                "config_name": config_name,
                "experiment_name": experiment_name,
                "repeat_idx": repeat_idx + 1,
                "seed": run_seed,
                "model_type": cfg_run.get("model_type"),
                "hidden_size": cfg_run.get("hidden_size"),
                "num_layers": cfg_run.get("num_layers"),
                "dropout": cfg_run.get("dropout"),
                "learning_rate": cfg_run.get("learning_rate"),
                "use_mlp_head": cfg_run.get("use_mlp_head"),
                "mlp_hidden_size": cfg_run.get("mlp_hidden_size"),
                "lookback": cfg_run.get("lookback"),
                "horizon": cfg_run.get("horizon"),
                "num_attention_heads": cfg_run.get("num_attention_heads", np.nan),
                "demand_lags": str(cfg_run.get("demand_lags", [])),
                "temp_lags": str(cfg_run.get("temp_lags", [])),
                "best_epoch": result.get("best_epoch"),
                "best_val_loss": result.get("best_val_loss"),
                "test_rmse": eval_dict.get("rmse"),
                "test_mae": eval_dict.get("mae"),
                "test_mape": eval_dict.get("mape"),
                "test_r2": eval_dict.get("r2"),
                "test_acc_within_10pct": eval_dict.get("within_tol_acc"),
            }

            all_rows.append(row)
            all_artifacts.append(artifact)

    runs_df = pd.DataFrame(all_rows)

    metric_cols = [
        "best_val_loss",
        "test_rmse",
        "test_mae",
        "test_mape",
        "test_r2",
        "test_acc_within_10pct",
    ]
    for col in metric_cols:
        if col in runs_df.columns:
            runs_df[col] = pd.to_numeric(runs_df[col], errors="coerce")

    if not runs_df.empty and "best_val_loss" in runs_df.columns:
        runs_df = runs_df.sort_values(
            by=["best_val_loss", "test_rmse"],
            ascending=[True, True]
        ).reset_index(drop=True)

    if save_dir:
        runs_df.to_csv(os.path.join(save_dir, "all_runs_raw.csv"), index=False)

        # Save a lightweight JSON copy of row-wise results
        with open(os.path.join(save_dir, "all_runs_raw.json"), "w") as f:
            json.dump(runs_df.to_dict(orient="records"), f, indent=2)

    return runs_df, all_artifacts


def summarise_runs(runs_df):
    metric_cols = [
        "best_val_loss",
        "test_rmse",
        "test_mae",
        "test_mape",
        "test_r2",
        "test_acc_within_10pct",
    ]

    group_cols = [
        "config_name",
        "model_type",
        "hidden_size",
        "num_layers",
        "dropout",
        "learning_rate",
        "use_mlp_head",
        "mlp_hidden_size",
        "lookback",
        "horizon",
        "demand_lags",
        "temp_lags",
        "num_attention_heads",
    ]

    # only keep columns that actually exist
    group_cols = [c for c in group_cols if c in runs_df.columns]

    summary_rows = []

    grouped = runs_df.groupby(group_cols, dropna=False)
    for group_key, group_df in grouped:
        row = dict(zip(group_cols, group_key))
        n = len(group_df)
        row["n_runs"] = n

        for metric in metric_cols:
            if metric not in group_df.columns:
                row[f"{metric}_mean"] = np.nan
                row[f"{metric}_std"] = np.nan
                row[f"{metric}_sem"] = np.nan
                row[f"{metric}_ci95_low"] = np.nan
                row[f"{metric}_ci95_high"] = np.nan
                continue

            vals = group_df[metric].dropna().astype(float).values

            if len(vals) == 0:
                row[f"{metric}_mean"] = np.nan
                row[f"{metric}_std"] = np.nan
                row[f"{metric}_sem"] = np.nan
                row[f"{metric}_ci95_low"] = np.nan
                row[f"{metric}_ci95_high"] = np.nan
                continue

            mean_val = float(np.mean(vals))
            std_val = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            sem_val = std_val / math.sqrt(len(vals)) if len(vals) > 1 else 0.0
            ci95 = 1.96 * sem_val

            row[f"{metric}_mean"] = mean_val
            row[f"{metric}_std"] = std_val
            row[f"{metric}_sem"] = sem_val
            row[f"{metric}_ci95_low"] = mean_val - ci95
            row[f"{metric}_ci95_high"] = mean_val + ci95

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    if not summary_df.empty and "test_rmse_mean" in summary_df.columns:
        summary_df = summary_df.sort_values(
            by=["test_rmse_mean", "test_mae_mean"],
            ascending=[True, True]
        ).reset_index(drop=True)

    return summary_df


def get_best_artifact(runs_df, all_artifacts):
    if runs_df.empty:
        raise ValueError("runs_df is empty.")

    best_experiment_name = runs_df.iloc[0]["experiment_name"]
    best_artifact = next(
        a for a in all_artifacts
        if a["result"]["experiment_name"] == best_experiment_name
    )
    return best_artifact


def plot_best_run(runs_df, all_artifacts, n_points=500):
    best_artifact = get_best_artifact(runs_df, all_artifacts)
    best_experiment_name = best_artifact["result"]["experiment_name"]

    print("Best experiment:", best_experiment_name)
    print(runs_df.iloc[0])

    plot_training_history(
        best_artifact["train_losses"],
        best_artifact["val_losses"],
        title=f"Training History - {best_experiment_name}",
    )

    plot_predictions(
        best_artifact["eval_dict"]["y_true_real"],
        best_artifact["eval_dict"]["y_pred_real"],
        n_points=n_points,
        title=f"Forecast vs Actual - {best_experiment_name}",
    )

    return best_artifact


def plot_attention_for_sample(artifact, sample_index=0):
    if artifact["config"]["model_type"] not in ["attention_bilstm", "multihead_attention_bilstm"]:
        print("Selected artifact is not an attention model.")
        return

    config = artifact["config"]
    data_dict = load_data(config)
    seq_dict = build_dataloaders(data_dict, config)

    model = build_model(config, input_size=artifact["input_size"])
    model.load_state_dict(artifact["best_state"])
    model.eval()

    X_test = torch.tensor(seq_dict["X_test"], dtype=torch.float32)

    if sample_index < 0 or sample_index >= len(X_test):
        raise IndexError(f"sample_index must be between 0 and {len(X_test)-1}")

    with torch.no_grad():
        _, attn_weights = model(
            X_test[sample_index:sample_index + 1].to(DEVICE),
            return_attention=True,
        )

    weights = attn_weights.squeeze(0).cpu().numpy()

    plt.figure(figsize=(10, 4))

    if weights.ndim == 1:
        # single-head attention_bilstm
        plt.plot(weights, label="Attention")
    else:
        # multihead_attention_bilstm -> shape (num_heads, seq_len)
        for h in range(weights.shape[0]):
            plt.plot(weights[h], label=f"Head {h+1}")

    plt.xlabel("Timestep in lookback window")
    plt.ylabel("Attention weight")
    plt.title(f"Attention weights - sample {sample_index}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_metric_distribution(runs_df, metric="test_rmse", title=None):
    vals = runs_df[metric].dropna().astype(float).values

    if len(vals) == 0:
        print(f"No values found for {metric}")
        return

    plt.figure(figsize=(7, 5))
    plt.boxplot(vals, widths=0.4)

    x_jitter = np.random.normal(1, 0.03, size=len(vals))
    plt.scatter(x_jitter, vals, alpha=0.6)

    plt.title(title or f"{metric} Distribution")
    plt.ylabel(metric)
    plt.xticks([1], ["Config"])
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_metric_distribution_with_summary(runs_df, metric="test_rmse", title=None):
    vals = runs_df[metric].dropna().astype(float).values
    mae_vals = runs_df["test_mae"].dropna().astype(float).values if "test_mae" in runs_df.columns else np.array([])
    r2_vals = runs_df["test_r2"].dropna().astype(float).values if "test_r2" in runs_df.columns else np.array([])
    acc_vals = runs_df["test_acc_within_10pct"].dropna().astype(float).values if "test_acc_within_10pct" in runs_df.columns else np.array([])

    if len(vals) == 0:
        print(f"No values found for {metric}")
        return

    plt.figure(figsize=(7, 5))
    plt.boxplot(vals, widths=0.4)

    x_jitter = np.random.normal(1, 0.03, size=len(vals))
    plt.scatter(x_jitter, vals, alpha=0.6)

    plt.title(title or f"{metric} Distribution")
    plt.ylabel(metric)
    plt.xticks([1], ["Config"])
    plt.grid(True, axis="y", alpha=0.3)

    text_parts = []
    if len(mae_vals) > 0:
        text_parts.append(f"MAE: {mae_vals.mean():.2f} ± {mae_vals.std():.2f}")
    if len(r2_vals) > 0:
        text_parts.append(f"R²: {r2_vals.mean():.3f} ± {r2_vals.std():.3f}")
    if len(acc_vals) > 0:
        text_parts.append(f"Acc (±10%): {acc_vals.mean():.2f} ± {acc_vals.std():.2f}")

    if text_parts:
        plt.gca().text(
            1.02, 0.95,
            "\n".join(text_parts),
            transform=plt.gca().transAxes,
            fontsize=10,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

    plt.tight_layout()
    plt.show()

In [ ]:
RUN_SINGLE = False
RUN_SWEEP = True
RUN_REPEATED = False

# SINGLE EXPERIMENT
if RUN_SINGLE:
    runs_df, all_artifacts = run_experiment_suite(
        base_config=BASE_CONFIG,
        param_grid=None,
        n_repeats=1,
        base_seed=SEED,
        save_dir=OUTPUT_DIR if SAVE_RESULTS else None,
        save_best_model=SAVE_BEST_MODEL,
        show_live_plots=SHOW_LIVE_PLOTS,
    )

    summary_df = summarise_runs(runs_df)

    print("\nRun results:")
    display(runs_df)

    print("\nSummary:")
    display(summary_df)

    best_artifact = plot_best_run(runs_df, all_artifacts, n_points=500)

    if best_artifact["config"]["model_type"] in ["attention_bilstm", "multihead_attention_bilstm"]:
        plot_attention_for_sample(best_artifact, sample_index=0)

if RUN_SWEEP:

    # PARAMETER SWEEP

    PARAM_GRID = {
        "lookback": [168, 336, 672],
        "horizon": [336],
        "model_type": ["lstm", "bilstm", "multihead_attention_bilstm"],
        "hidden_size": [64],
        "num_layers": [2],
        "dropout": [0.1],
        "learning_rate": [5e-5],
    }

    runs_df, all_artifacts = run_experiment_suite(
        base_config=BASE_CONFIG,
        param_grid=PARAM_GRID,
        n_repeats=1,
        base_seed=SEED,
        save_dir=OUTPUT_DIR if SAVE_RESULTS else None,
        save_best_model=False,
        show_live_plots=SHOW_LIVE_PLOTS,
    )

    summary_df = summarise_runs(runs_df)

    print("\nRaw run results:")
    display(runs_df.head())

    print("\nSummary:")
    display(summary_df)

    best_artifact = plot_best_run(runs_df, all_artifacts, n_points=500)

    if best_artifact["config"]["model_type"] in ["attention_bilstm", "multihead_attention_bilstm"]:
        plot_attention_for_sample(best_artifact, sample_index=0)

if RUN_REPEATED:
    # REPEATED EXPERIMENTS

    REPEAT_N = 5
    SAVE_DIR = "repeated_experiment_outputs"

    PARAM_GRID = {
        "lookback": [168, 336, 672],
        "horizon": [336],
        "model_type": ["lstm", "bilstm", "multihead_attention_bilstm"],
        "hidden_size": [64],
        "num_layers": [2],
        "dropout": [0.1],
        "learning_rate": [5e-5],
    }

    runs_df, all_artifacts = run_experiment_suite(
            base_config=BASE_CONFIG,
            param_grid=PARAM_GRID,
            n_repeats=REPEAT_N,
            base_seed=1000,
            save_dir=SAVE_DIR,
            save_best_model=False,
            show_live_plots=False,
    )

    summary_df = summarise_runs(runs_df)

    runs_df.to_csv(os.path.join(SAVE_DIR, "all_runs_raw.csv"), index=False)
    summary_df.to_csv(os.path.join(SAVE_DIR, "summary_stats.csv"), index=False)

    print("\nRaw run results:")
    display(runs_df.head())

    print("\nSummary results:")
    display(summary_df)

    plot_metric_distribution_with_summary(
        runs_df,
        metric="test_rmse",
        title=f"Test RMSE Distribution ({len(runs_df)} Runs)"
    )



Total configurations: 9
Repeats per configuration: 1
Total runs: 9

[CONFIG 1/9] lstm_lb168_hs64_nl2_do0.1_lr5e-05_mlp1_dlagsnone_tlags0-2-50_heads4

Run 1/9 | repeat 1/1 | seed=42

Running experiment: lstm_lb168_hs64_nl2_do0.1_lr5e-05_mlp1_dlagsnone_tlags0-2-50_heads4
model_type: lstm
hidden_size: 64
num_layers: 2
dropout: 0.1
learning_rate: 5e-05
batch_size: 64
epochs: 100
patience: 8
use_mlp_head: True
mlp_hidden_size: 64
target_col: TOTALDEMAND
temp_col: TEMPERATURE
demand_lags: []
temp_lags: [0, 2, 50]
lookback: 168
horizon: 336
weight_decay: 0.0001
num_attention_heads: 4
seed: 42
Using run seed: 42

DEBUG: FEATURE INSPECTION

No feature list found in data_dict

Sequence shape (lookback, num_features): (168, 5)

Raw sequence values (first 5 timesteps):
[[ 1.0609382  -0.39498806 -0.39498806 -0.50004727 -1.0778731 ]
 [ 0.91395706 -0.4124979  -0.4124979  -0.46502754 -0.9903237 ]
 [ 0.8413571  -0.37747818 -0.37747818 -0.39498806 -0.97281384]
 [ 0.73133713 -0.3599683  -0.3599683  -0.41